# Comparaison de sept méthodes avec un protocole commun

**Objectif :** comparer, dans exactement les mêmes conditions, sept méthodes de filtrage collaboratif :

| # | Méthode | Source |
|---|---|---|
| 1 | Voisinage (k plus proches voisins utilisateurs) | cours, telle quelle |
| 2 | Factorisation de matrice par descente de gradient | cours, telle quelle |
| 3 | SVD++ : factorisation + biais + information implicite | Koren (2008), *Factorization Meets the Neighborhood* |
| 4 | Factorisation bayésienne : factorisation + biais, apprise par MCMC | ligne « Bayesian MF » de Rendle, Zhang, Koren (2019) |
| 5 | Bayesian SVD++ : la réunion de 3 et 4 | Rendle, Zhang, Koren (2019), *On the Difficulty of Evaluating Baselines* |
| 6 | Bayesian SVD++ *flipped* : la méthode 5, avec l'information implicite aussi côté film | variante « flipped » de Rendle, Zhang, Koren (2019), sans le temps |
| 7 | Méthode 6, en moyennant plusieurs chaînes MCMC | — |

Les méthodes 3 et 4 isolent chacune l'un des deux ingrédients de la méthode 5 : on voit ce que chacun apporte seul, puis ensemble. La méthode 6 a été ajoutée **après** avoir vu le résultat de la méthode 5 ; la section 9 explique pourquoi. La méthode 7 calcule la méthode 6 avec plusieurs chaînes MCMC (section 10).

**Pour chaque méthode, on obtient :**
1. les meilleurs hyperparamètres, cherchés par **Optuna** en **validation croisée à 5 plis** sur le train ;
2. le RMSE de validation (moyenne sur les 5 plis) ;
3. le RMSE de test, **mesuré une seule fois**.

**Le protocole, identique pour les sept méthodes :**

| Étape | Ce qu'on fait | Données |
|---|---|---|
| 1 | Découper le train en 5 plis (les **mêmes** pour toutes les méthodes) | train |
| 2 | Pour chaque réglage proposé par Optuna : apprendre sur 4 plis, mesurer sur le 5ᵉ, 5 fois ; score = moyenne | train |
| 3 | Garder le réglage de meilleur score (même nombre d'essais pour chaque méthode) | train |
| 4 | Réapprendre ce réglage sur **tout** le train, mesurer **une fois** sur le test | train → test |

Le test n'est lu qu'à l'étape 4.

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["TQDM_DISABLE"] = "1"                 # pas de barres de progression de myFM
import numpy as np
import scipy.sparse as sp
import optuna
import myfm

SEED, N_FOLDS, N_TRIALS = 0, 5, 30
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Données et mesure d'erreur

Notations du cours : $R\in\mathbb R^{m\times n}$, une ligne par **film** $i$, une colonne par **utilisateur** $u$. Les fichiers sont rangés utilisateurs × films, on les transpose. `NaN` = note inconnue.

L'erreur est le RMSE du cours, calculé seulement sur les cases à prédire $T$ :
$$\mathrm{RMSE}=\sqrt{\tfrac{1}{|T|}\textstyle\sum_{(i,u)\in T}\big(R_{iu}-\widehat R_{iu}\big)^2}$$
Deux conventions, les mêmes pour toutes les méthodes :
- les prédictions sont ramenées dans l'intervalle des notes, $[0{,}5\,;\,5]$ ;
- une prédiction invalide (calcul qui a divergé) compte comme 0, donc comme une très mauvaise prédiction.

In [ ]:
R_train = np.load("ratings_train.npy").T      # films × utilisateurs
R_test  = np.load("ratings_test.npy").T       # lu ici, utilisé seulement à l'étape 4
m, n = R_train.shape
print(f"{m} films, {n} utilisateurs | train : {np.sum(~np.isnan(R_train))} notes | test : {np.sum(~np.isnan(R_test))} notes")

def rmse(P, R_true):
    T = ~np.isnan(R_true)
    p = np.clip(np.nan_to_num(P[T], nan=0.0, posinf=5, neginf=0.5), 0.5, 5)
    return float(np.sqrt(np.mean((p - R_true[T]) ** 2)))

4980 films, 610 utilisateurs | train : 31598 notes | test : 31598 notes


## 2. Les 5 plis de validation croisée

Chaque note du train est tirée au hasard dans l'un des 5 plis. Pour le pli $f$ : on **apprend** sur les notes des 4 autres plis et on **mesure** sur celles du pli $f$.

**Une contrainte :** chaque film et chaque utilisateur doit garder au moins une note dans la partie apprentissage, sinon on demanderait de prédire un film jamais vu. On réserve donc, pour chaque film et chaque utilisateur, une note tirée au hasard, qui reste **toujours** en apprentissage et n'est jamais mesurée.

Les plis sont construits **une seule fois**, avec une graine fixe : les sept méthodes sont évaluées sur exactement les mêmes cases.

In [ ]:
rng = np.random.default_rng(SEED)
ii, uu = np.where(~np.isnan(R_train))
fold = rng.integers(0, N_FOLDS, len(ii))
for idx in (ii, uu):                                   # une note réservée par film et par utilisateur
    for e in np.unique(idx):
        fold[rng.choice(np.where(idx == e)[0])] = -1

FOLDS = []
for f in range(N_FOLDS):
    fit, val = R_train.copy(), np.full_like(R_train, np.nan)
    fit[ii[fold == f], uu[fold == f]] = np.nan
    val[ii[fold == f], uu[fold == f]] = R_train[ii[fold == f], uu[fold == f]]
    FOLDS.append((fit, val))
print("notes réservées (toujours en apprentissage) :", np.sum(fold == -1))
print("notes de validation par pli :", [int(np.sum(~np.isnan(v))) for _, v in FOLDS])

notes réservées (toujours en apprentissage) : 5529
notes de validation par pli : [5326, 5150, 5278, 5101, 5214]


## 3. La procédure commune

Chaque méthode est écrite sous la même forme : `predict(R, R_targets, **hp)` apprend sur les notes connues de `R` et renvoie une matrice de prédictions (au moins sur les cases de `R_targets`).

La fonction `run` applique le protocole à n'importe quelle méthode :
- **Optuna** propose `N_TRIALS` réglages, avec l'échantillonneur TPE (il propose de nouveaux réglages là où les précédents ont bien marché). La graine est fixée.
- **Score d'un réglage** = RMSE moyen sur les 5 plis.
- **Puis** : réapprentissage du meilleur réglage sur tout le train, et **une seule** mesure sur le test.

In [ ]:
RESULTS = {}

def run(name, predict, space, sampler=None, n_trials=N_TRIALS):
    def objective(trial):
        hp = space(trial)
        scores = [rmse(predict(fit, val, **hp), val) for fit, val in FOLDS]
        trial.set_user_attr("std", float(np.std(scores)))
        return float(np.mean(scores))

    t0 = time.time()
    study = optuna.create_study(direction="minimize", sampler=sampler or optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    t_cv = time.time() - t0

    t0 = time.time()
    test = rmse(predict(R_train, R_test, **study.best_params), R_test)      # la seule mesure sur le test
    RESULTS[name] = dict(best=study.best_params, val=study.best_value, std=study.best_trial.user_attrs["std"],
                         test=test, t_cv=t_cv, t_fit=time.time() - t0)
    r = RESULTS[name]
    print(f"{name}\n  meilleurs hyperparamètres : {r['best']}\n"
          f"  RMSE validation (5 plis) : {r['val']:.4f} ± {r['std']:.4f}\n  RMSE test : {r['test']:.4f}\n"
          f"  temps : recherche {r['t_cv']/60:.1f} min, apprentissage final {r['t_fit']:.0f} s")

## 4. Méthode 1 — Voisinage (cours)

**L'algorithme du cours**, pour prédire $\widehat R_{iu}$ :
1. calculer $\mathrm{Sim}(u,v)$ pour tous les utilisateurs $v$ ;
2. garder au plus $k$ plus proches voisins **qui ont noté le film $i$** : $N_k(u;i)$ ;
3. prédire
$$\widehat R_{iu}=\frac{\sum_{v\in N_k(u;i)}\mathrm{Sim}(u,v)\,R_{iv}}{\sum_{v\in N_k(u;i)}\lvert\mathrm{Sim}(u,v)\rvert},$$
et utiliser une **valeur de repli** si le dénominateur est nul.

**Les deux similarités du cours :**
- **Jaccard** sur les films aimés, $X_u=\{i : R_{iu}\ge 3\}$ : $\;\mathrm{Sim}(u,v)=\dfrac{|X_u\cap X_v|}{|X_u\cup X_v|}$ ;
- **Pearson** sur les films notés par les deux, $M_u\cap M_v$, avec $\mu_u$ la moyenne des notes de $u$ :
$$\mathrm{Sim}(u,v)=\frac{\sum_{k\in M_u\cap M_v}(r_{ku}-\mu_u)(r_{kv}-\mu_v)}{\sqrt{\sum_{k\in M_u\cap M_v}(r_{ku}-\mu_u)^2}\,\sqrt{\sum_{k\in M_u\cap M_v}(r_{kv}-\mu_v)^2}}$$

**Ce que le cours ne fixe pas, et ce qu'on a choisi :**
- la valeur de repli : $\mu_u$, la moyenne de l'utilisateur (déjà définie dans le cours pour Pearson) ;
- une similarité non définie (aucun film en commun, ou dénominateur nul) vaut 0.

**Hyperparamètres cherchés :** $k\in[1,200]$ et la similarité (Jaccard ou Pearson).

**Calcul :** les similarités entre les 610 utilisateurs tiennent dans une matrice $610\times610$, calculée par produits de matrices.

In [ ]:
def similarity(R, kind):
    M = ~np.isnan(R)
    if kind == "jaccard":
        X = (np.nan_to_num(R) >= 3).astype(float)                  # X_u = films notés >= 3
        inter = X.T @ X
        union = X.sum(0)[:, None] + X.sum(0)[None, :] - inter
        return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
    Dc = np.where(M, R - np.nanmean(R, 0), 0.0)                    # r_ku - mu_u sur les films notés, 0 sinon
    num = Dc.T @ Dc                                                # somme sur M_u ∩ M_v
    A = (Dc ** 2).T @ M                                            # A[u, v] = somme sur M_u ∩ M_v de (r_ku - mu_u)^2
    den = np.sqrt(A * A.T)
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def knn(R, R_targets, k, sim):
    S, M, mu = similarity(R, sim), ~np.isnan(R), np.nanmean(R, 0)
    P = np.full(R.shape, np.nan)
    for i, u in zip(*np.where(~np.isnan(R_targets))):
        v = np.where(M[i])[0]                                      # utilisateurs qui ont noté i
        v = v[np.argsort(-S[u, v])[:k]]                            # les k plus similaires à u
        s = S[u, v]
        P[i, u] = s @ R[i, v] / np.abs(s).sum() if np.abs(s).sum() > 0 else mu[u]
    return P

run("Voisinage (cours)", knn,
    lambda t: dict(k=t.suggest_int("k", 1, 200, log=True), sim=t.suggest_categorical("sim", ["jaccard", "pearson"])))

Voisinage (cours)
  meilleurs hyperparamètres : {'k': 144, 'sim': 'jaccard'}
  RMSE validation (5 plis) : 0.9944 ± 0.0090
  RMSE test : 1.0060
  temps : recherche 0.5 min, apprentissage final 1 s


**Lecture :** Optuna retient **Jaccard avec $k=144$** : 0,994 en validation, 1,006 en test. C'est **moins bien que de prédire la moyenne de chaque utilisateur** (0,945 en test dans nos autres notebooks). Deux raisons, qui viennent des données et de la formule :
- **Un film médian n'a que 3 notes.** Les « $k$ plus proches voisins qui ont noté $i$ » sont donc presque toujours *tous* ceux qui l'ont noté : avec $k=144$, la limite ne joue quasiment jamais. La prédiction devient une moyenne des notes du film, pondérée par la similarité. Elle ne tient pas compte du fait que $u$ note plus sévèrement ou plus généreusement que ses voisins.
- **Pearson est écarté** parce que ses similarités peuvent être négatives. La formule du cours donne alors un poids négatif à une note brute (par exemple $-0{,}8\times5$), ce qui envoie la prédiction hors de l'échelle des notes. Cette formule suppose des similarités positives, ou des notes centrées.

## 5. Méthode 2 — Factorisation de matrice (cours)

**Le modèle du cours :** $R\approx I\,U^\top$, avec $I\in\mathbb R^{m\times k}$ (un vecteur par film) et $U\in\mathbb R^{n\times k}$ (un vecteur par utilisateur).

**Le coût, sur les seules cases observées $S$** (diapositive « Missing values ») :
$$C(I,U)=\sum_{(i,u)\in S}\Big(R_{iu}-\sum_{s=1}^k I_{is}U_{us}\Big)^2+\lambda\lVert I\rVert_F^2+\mu\lVert U\rVert_F^2$$

**Ses dérivées.** Notons $E$ la matrice des erreurs $R_{iu}-(IU^\top)_{iu}$ sur $S$, et 0 ailleurs :
$$\frac{\partial C}{\partial I}=-2\,E\,U+2\lambda I,\qquad \frac{\partial C}{\partial U}=-2\,E^\top I+2\mu U$$

**L'algorithme : la descente de gradient du cours**, avec un pas constant $\eta$ (le même pour $I$ et $U$) :
$$I_{t+1}=I_t-\eta\,\frac{\partial C}{\partial I}(I_t,U_t),\qquad U_{t+1}=U_t-\eta\,\frac{\partial C}{\partial U}(I_t,U_t)$$
(L'ALS du cours est écrite pour une matrice complète ; avec des cases manquantes, le cours donne les dérivées, d'où le choix de la descente de gradient.)

**Ce que le cours ne fixe pas, et ce qu'on a choisi :** le point de départ, des valeurs tirées selon $\mathcal N(0;\,0{,}1^2)$.

**Hyperparamètres cherchés :** $k\in[1,64]$, $\lambda$ et $\mu\in[10^{-3},10^2]$, $\eta\in[10^{-4},5\cdot10^{-3}]$, nombre d'itérations $\in[100,3000]$. Au-delà de $\eta\approx3\cdot10^{-3}$, la descente diverge ; Optuna le constate et s'en écarte.

**Calcul :** les erreurs ne sont calculées que sur les cases observées (matrice creuse $E$).

In [ ]:
def mf(R, R_targets, k, lam, mu, eta, n_iter):
    rng = np.random.default_rng(SEED)
    ii, uu = np.where(~np.isnan(R)); r = R[ii, uu]
    I = 0.1 * rng.standard_normal((m, k)); U = 0.1 * rng.standard_normal((n, k))
    with np.errstate(all="ignore"):                                # une divergence donne des NaN, comptés comme erreurs
        for _ in range(n_iter):
            E = sp.csr_matrix((r - np.einsum("ij,ij->i", I[ii], U[uu]), (ii, uu)), shape=(m, n))
            I, U = I - eta * (-2 * E @ U + 2 * lam * I), U - eta * (-2 * E.T @ I + 2 * mu * U)
        return I @ U.T

run("Factorisation (cours)", mf,
    lambda t: dict(k=t.suggest_int("k", 1, 64, log=True),
                   lam=t.suggest_float("lam", 1e-3, 1e2, log=True), mu=t.suggest_float("mu", 1e-3, 1e2, log=True),
                   eta=t.suggest_float("eta", 1e-4, 5e-3, log=True), n_iter=t.suggest_int("n_iter", 100, 3000, log=True)))

Factorisation (cours)
  meilleurs hyperparamètres : {'k': 1, 'lam': 0.014684803764875042, 'mu': 0.9492861281053446, 'eta': 0.0002730449359463328, 'n_iter': 694}
  RMSE validation (5 plis) : 0.9049 ± 0.0086
  RMSE test : 0.9123
  temps : recherche 4.0 min, apprentissage final 1 s


**Lecture :** Optuna retient **$k=1$**, $\lambda\approx0{,}015$, $\mu\approx0{,}95$, $\eta\approx2{,}7\cdot10^{-4}$, 694 itérations : 0,905 en validation, 0,912 en test. C'est nettement mieux que le voisinage.
- **$k=1$ : un seul nombre par film et par utilisateur**, et la prédiction est leur produit $I_iU_u$. Le modèle du cours n'a pas de terme de moyenne : ses facteurs doivent d'abord produire le niveau moyen des notes (≈ 3,5). Une dimension suffit pour cela, et chaque dimension en plus ajoute des paramètres qui sur-apprennent les films peu notés.
- **Le pas est petit et le nombre d'itérations limité** : la descente n'est pas allée jusqu'au minimum du coût. S'arrêter avant joue le rôle d'une régularisation, ce qui explique qu'un $\lambda$ aussi petit convienne.

## 6. Méthode 3 — SVD++ (Koren 2008)

**Le modèle** (notations du cours : $q_i=I_i$, $p_u=U_u$) :
$$\widehat R_{iu}=\mu+b_i+b_u+\Big\langle q_i\,,\;p_u+\lvert N(u)\rvert^{-1/2}\textstyle\sum_{j\in N(u)}y_j\Big\rangle$$
- $\mu$ : moyenne de toutes les notes ; $b_i$, $b_u$ : biais du film et de l'utilisateur ;
- $\langle q_i,p_u\rangle$ : la factorisation du cours ;
- $N(u)$ : les films **notés** par $u$ dans les données d'entrée, quelle que soit la note. Chaque film $j$ a un second vecteur $y_j$, qui dit ce qu'avoir noté $j$ révèle du goût.

**L'apprentissage de l'article : descente de gradient stochastique.** On parcourt les notes une par une (utilisateurs dans un ordre aléatoire, puis leurs notes dans un ordre aléatoire). Pour chaque note, on calcule l'erreur $e=R_{iu}-\widehat R_{iu}$, puis, avec $z_u=p_u+|N(u)|^{-1/2}\sum_{j\in N(u)}y_j$ :

| Paramètre | Mise à jour |
|---|---|
| $b_u$, $b_i$ | $b\leftarrow b+\gamma\,(e-\lambda_b\,b)$ |
| $q_i$ | $q_i\leftarrow q_i+\gamma\,(e\,z_u-\lambda\,q_i)$ |
| $p_u$ | $p_u\leftarrow p_u+\gamma\,(e\,q_i-\lambda\,p_u)$ |
| $y_j$, pour tous les $j\in N(u)$ | $y_j\leftarrow y_j+\gamma\,(e\,|N(u)|^{-1/2}q_i-\lambda\,y_j)$ |

Après chaque passage sur toutes les notes (une **époque**), le pas $\gamma$ est multiplié par 0,9, comme dans l'article.

**Ce que l'article ne fixe pas, et ce qu'on a choisi :** le point de départ, $\mathcal N(0;\,0{,}1^2)$ pour $p_u$, $q_i$, $y_j$ (comme les autres méthodes), et des biais à 0.

**Hyperparamètres cherchés :** $k\in[2,128]$, $\gamma\in[10^{-3},3\cdot10^{-2}]$, $\lambda_b$ et $\lambda\in[10^{-3},1]$, nombre d'époques $\in[5,60]$. Les valeurs de l'article ($\gamma=0{,}007$, $\lambda_b=0{,}005$, $\lambda=0{,}015$) sont dans cet espace.

**Calcul :** la boucle sur les notes est compilée avec `numba` (même algorithme, environ 50 fois plus rapide qu'en Python pur).

In [ ]:
from numba import njit

@njit
def svdpp_sgd(ptr, items, ratings, mu, bu, bi, P, Q, Y, lr, reg_b, reg, n_epochs):
    np.random.seed(SEED)
    k = P.shape[1]
    for _ in range(n_epochs):
        for u in np.random.permutation(len(bu)):
            its, rs = items[ptr[u]:ptr[u + 1]], ratings[ptr[u]:ptr[u + 1]]      # N(u) et les notes de u
            c = 1.0 / np.sqrt(max(len(its), 1))
            S = np.zeros(k)
            for j in its: S += Y[j]                                            # somme des y_j, tenue à jour
            for t in np.random.permutation(len(its)):
                i = its[t]
                z = P[u] + c * S
                e = rs[t] - (mu + bu[u] + bi[i] + Q[i] @ z)
                bu[u] += lr * (e - reg_b * bu[u]); bi[i] += lr * (e - reg_b * bi[i])
                qi = Q[i].copy()
                Q[i] += lr * (e * z - reg * Q[i]); P[u] += lr * (e * qi - reg * P[u])
                for j in its:
                    d = lr * (e * c * qi - reg * Y[j]); Y[j] += d; S += d
        lr *= 0.9

def svdpp(R, R_targets, k, lr, reg_b, reg, n_epochs):
    rng = np.random.default_rng(SEED)
    Nu = sp.csc_matrix(~np.isnan(R))                                     # colonne u : films notés par u
    ptr, items = Nu.indptr, Nu.indices
    ratings = R[items, np.repeat(np.arange(n), np.diff(ptr))]
    mu, bu, bi = float(np.nanmean(R)), np.zeros(n), np.zeros(m)
    P, Q, Y = (0.1 * rng.standard_normal((s, k)) for s in (n, m, m))
    with np.errstate(all="ignore"):
        svdpp_sgd(ptr, items, ratings, mu, bu, bi, P, Q, Y, lr, reg_b, reg, n_epochs)
        Z = P + sp.diags(1 / np.sqrt(np.maximum(np.diff(ptr), 1))) @ Nu.T @ Y    # z_u pour chaque utilisateur
        return mu + bi[:, None] + bu[None, :] + Q @ Z.T

run("SVD++ (Koren 2008)", svdpp,
    lambda t: dict(k=t.suggest_int("k", 2, 128, log=True), lr=t.suggest_float("lr", 1e-3, 3e-2, log=True),
                   reg_b=t.suggest_float("reg_b", 1e-3, 1, log=True), reg=t.suggest_float("reg", 1e-3, 1, log=True),
                   n_epochs=t.suggest_int("n_epochs", 5, 60)))

SVD++ (Koren 2008)
  meilleurs hyperparamètres : {'k': 5, 'lr': 0.026036270993428392, 'reg_b': 0.10947980755691644, 'reg': 0.19484181194855843, 'n_epochs': 49}
  RMSE validation (5 plis) : 0.8712 ± 0.0107
  RMSE test : 0.8767
  temps : recherche 38.0 min, apprentissage final 25 s


**Lecture :** Optuna retient **$k=5$, $\gamma\approx0{,}026$, $\lambda_b\approx0{,}11$, $\lambda\approx0{,}19$, 49 époques** : 0,871 en validation, 0,877 en test.
- C'est **0,036 de mieux que la factorisation du cours** (0,912). Le gain vient des biais et de l'information implicite, que la factorisation du cours n'a pas.
- **Les réglages sont loin de ceux de l'article** : une régularisation environ 10 fois plus forte ($\lambda\approx0{,}19$ contre 0,015) et seulement 5 dimensions. L'article était réglé sur Netflix, 3 000 fois plus de notes. Ici, un film médian n'a que 3 notes : il faut beaucoup régulariser pour ne pas les apprendre par cœur.
- L'écart entre validation et test (0,0055) est un peu plus grand que pour les autres méthodes.

## 7. Méthode 4 — Factorisation bayésienne

**Le modèle :** la factorisation du cours, avec des biais :
$$\widehat R_{iu}=w_0+w_u+w_i+\langle v_u,v_i\rangle$$

**L'apprentissage bayésien.** Au lieu de chercher *un* jeu de paramètres, on en **tire** un à chaque itération par échantillonnage de Gibbs (MCMC), et on **moyenne** les prédictions de tous les tirages. Les forces de régularisation ne sont pas des hyperparamètres : elles sont tirées elles aussi, à partir des données, une par groupe de variables (utilisateurs, films).

C'est la ligne « Bayesian MF » de l'article de Rendle, Zhang et Koren (2019), apprise avec le même logiciel et les mêmes réglages que la méthode 5. **Le code est celui de la méthode 5, sans le bloc $N(u)$.**

**Hyperparamètres cherchés :** les mêmes que pour la méthode 5, rang parmi 16, 32, 64, 128 et nombre d'itérations dans $[64,512]$.

In [ ]:
def bayes_mf(R, R_targets, rank, n_iter):
    user_vars, item_vars = sp.identity(n, format="csr"), sp.identity(m, format="csr")    # [ identifiant u ], [ identifiant i ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(~np.isnan(R))
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=SEED)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter, group_shapes=[n, m])
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Factorisation bayésienne", bayes_mf,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Factorisation bayésienne
  meilleurs hyperparamètres : {'rank': 16, 'n_iter': 128}
  RMSE validation (5 plis) : 0.8636 ± 0.0110
  RMSE test : 0.8667
  temps : recherche 37.2 min, apprentissage final 5 s


**Lecture :** Optuna retient **le rang 16 et 128 itérations** : 0,864 en validation, **0,867 en test**.
- C'est **0,010 de mieux que SVD++, sans l'information implicite.** Sur ces données, apprendre de façon bayésienne (moyenner beaucoup de modèles plausibles et apprendre la régularisation) compte plus qu'ajouter l'information implicite.
- C'est aussi **très rapide** : 5 s pour l'apprentissage final.

## 8. Méthode 5 — Bayesian SVD++ (Rendle, Zhang, Koren 2019)

**Le modèle.** C'est la réunion des méthodes 3 et 4 : le modèle de SVD++, appris comme la factorisation bayésienne. Concrètement, c'est SVD++ écrit comme une *factorization machine*. Chaque note est décrite par trois groupes de variables (tableau 3 de l'article) :
- l'identifiant de l'utilisateur $u$ ;
- l'identifiant du film $i$ ;
- l'**information implicite** de l'utilisateur : l'ensemble $N(u)$ des films qu'il a notés, quelle que soit la note.

La prédiction est alors
$$\widehat R_{iu}=w_0+w_u+w_i+\Big\langle v_i\,,\;v_u+\lvert N(u)\rvert^{-1/2}\textstyle\sum_{j\in N(u)}y_j\Big\rangle\;(+\text{interactions entre }u\text{ et }N(u)),$$
c'est-à-dire la factorisation du cours, plus des biais, plus le terme implicite de SVD++.

**L'apprentissage bayésien.** Au lieu de chercher *un* jeu de paramètres, on en **tire** un à chaque itération par échantillonnage de Gibbs (MCMC), et on **moyenne** les prédictions de tous les tirages. Les forces de régularisation ne sont pas des hyperparamètres : elles sont elles-mêmes tirées à partir des données (une par groupe de variables).

**Fidélité à l'article :**

| Réglage | Article | Ici |
|---|---|---|
| Logiciel | libFM, représentation relationnelle | myFM, qui réimplémente le même échantillonneur, avec la même représentation relationnelle (`RelationBlock`) |
| Variables | $u$, $i$, implicite utilisateur | idem, un groupe de régularisation par type de variable |
| Initialisation | gaussienne, écart-type 0,1 | idem |
| Dimension | 16, 32, 64, 128 | cherchée parmi 16, 32, 64, 128 |
| Itérations | jusqu'à 512 | cherchées dans $[64, 512]$ |
| Normalisation de $N(u)$ | non précisée | $\lvert N(u)\rvert^{-1/2}$, celle de SVD++ |
| Contenu de $N(u)$ | films notés dans le train **et** le test (sans les notes du test) | films notés dans les **données d'entrée** seulement |

Sur le dernier point : sur la plateforme, on ne connaît pas les cases d'évaluation, on ne peut donc pas les mettre dans $N(u)$. On utilise la version stricte, que l'article dit à peine moins bonne.

In [ ]:
def bayes_svdpp(R, R_targets, rank, n_iter):
    M = ~np.isnan(R)
    Nu = sp.csr_matrix(M.T.astype(float))                                   # utilisateur -> films notés
    Nu = sp.diags(1 / np.sqrt(np.maximum(Nu.sum(1).A1, 1))) @ Nu             # normalisation |N(u)|^(-1/2)
    user_vars = sp.hstack([sp.identity(n, format="csr"), Nu]).tocsr()       # [ identifiant u | N(u) ]
    item_vars = sp.identity(m, format="csr")                                # [ identifiant i ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(M)
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=SEED)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter,
           group_shapes=[n, m, m])                                          # groupes : u, N(u), i
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Bayesian SVD++ (article)", bayes_svdpp,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Bayesian SVD++ (article)
  meilleurs hyperparamètres : {'rank': 32, 'n_iter': 246}
  RMSE validation (5 plis) : 0.8635 ± 0.0098
  RMSE test : 0.8667
  temps : recherche 94.7 min, apprentissage final 29 s


**Lecture :** Optuna retient **le rang 32 et 246 itérations** : 0,864 en validation, 0,867 en test. C'est **exactement le score de la factorisation bayésienne** (méthode 4) : 0,0001 d'écart en validation, 0,0000 en test, alors que l'écart-type entre plis est de 0,01.

**Ajouter l'information implicite côté utilisateur à la factorisation bayésienne n'apporte donc rien sur ces données.** C'est cohérent avec leur structure :
- les utilisateurs ont beaucoup de notes (médiane : 24). Leur vecteur est déjà bien estimé, et la liste de leurs films n'ajoute presque rien ;
- ce sont les **films** qui ont peu de notes (médiane : 3). L'information implicite aide quand on l'ajoute côté film : c'est la variante « flipped » de l'article : la méthode 6.

Le coût est plus élevé que pour la méthode 4 : 95 minutes de recherche contre 37, et 29 s d'apprentissage final contre 5 s.

## 9. Méthode 6 — Bayesian SVD++ *flipped*

### Pourquoi cette méthode

La méthode 5 ne fait **pas mieux** que la méthode 4 : ajouter à chaque utilisateur la liste des films qu'il a notés n'apporte rien. Les données expliquent pourquoi :

| | Nombre de notes dans le train (médiane) | Conséquence |
|---|---|---|
| Utilisateur | 24 | son vecteur est déjà bien estimé à partir de ses notes ; la liste de ses films n'ajoute presque rien |
| Film | 3 | son vecteur est très incertain ; toute information en plus compte |

L'information implicite devrait donc aider **côté film** : décrire un film aussi par **les utilisateurs qui l'ont noté**, quelle que soit la note. C'est exactement la variante « *flipped* » de l'article de Rendle, Zhang et Koren (2019), leur meilleure variante sur MovieLens 10M (0,7485, avec en plus le temps, absent de nos données).

### Le modèle

Le film reçoit le même traitement que l'utilisateur dans SVD++ : on ajoute au vecteur du film $v_i$ une somme de vecteurs $x_w$, un par utilisateur $w$ de l'ensemble $N(i)$ des utilisateurs qui ont noté $i$ :
$$\widehat R_{iu}=w_0+w_u+w_i+\Big\langle\, v_i+\lvert N(i)\rvert^{-1/2}\textstyle\sum_{w\in N(i)}x_w\;,\;\; v_u+\lvert N(u)\rvert^{-1/2}\sum_{j\in N(u)}y_j\Big\rangle\;(+\text{interactions à l'intérieur de chaque bloc})$$

**Le code est celui de la méthode 5, avec un bloc $N(i)$ en plus côté film**, normalisé de la même façon, et avec son propre groupe de régularisation. Même espace de recherche : rang parmi 16, 32, 64, 128, itérations dans $[64,512]$.

### Une précaution de méthode

Cette méthode a été choisie **après** avoir vu les résultats de validation des méthodes 4 et 5. Ce n'est pas un problème tant que le test reste fermé : le choix repose sur la validation croisée et sur la structure des données, et le test n'est mesuré qu'une fois, comme pour les autres.

In [ ]:
def bayes_flipped(R, R_targets, rank, n_iter, seed=SEED):
    M = ~np.isnan(R)
    Nu = sp.csr_matrix(M.T.astype(float))                                   # utilisateur -> films notés
    Nu = sp.diags(1 / np.sqrt(np.maximum(Nu.sum(1).A1, 1))) @ Nu             # |N(u)|^(-1/2)
    Ni = sp.csr_matrix(M.astype(float))                                     # film -> utilisateurs qui l'ont noté
    Ni = sp.diags(1 / np.sqrt(np.maximum(Ni.sum(1).A1, 1))) @ Ni             # |N(i)|^(-1/2)
    user_vars = sp.hstack([sp.identity(n, format="csr"), Nu]).tocsr()       # [ identifiant u | N(u) ]
    item_vars = sp.hstack([sp.identity(m, format="csr"), Ni]).tocsr()       # [ identifiant i | N(i) ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(M)
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=seed)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter,
           group_shapes=[n, m, m, n])                                       # groupes : u, N(u), i, N(i)
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Bayesian SVD++ flipped", bayes_flipped,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Bayesian SVD++ flipped
  meilleurs hyperparamètres : {'rank': 16, 'n_iter': 145}
  RMSE validation (5 plis) : 0.8527 ± 0.0106
  RMSE test : 0.8556
  temps : recherche 81.3 min, apprentissage final 10 s


**Lecture :** Optuna retient **le rang 16 et 145 itérations** : 0,853 en validation, **0,856 en test**. C'est la meilleure des méthodes 1 à 6, avec **0,011 de gain** sur les méthodes 4 et 5, aussi bien en validation qu'en test.

**L'hypothèse de départ est confirmée :** l'information implicite n'aide que là où les données manquent, c'est-à-dire côté film. Le coût reste faible : 10 s pour l'apprentissage final (81 minutes pour la recherche).

## 10. Méthode 7 — Méthode 6 en moyennant plusieurs chaînes MCMC

### Pourquoi cette méthode

La prédiction bayésienne est une **moyenne sur les tirages** d'une chaîne de Markov. Avec 145 itérations, cette moyenne garde une part de hasard : deux chaînes lancées avec des graines différentes ne donnent pas exactement les mêmes prédictions. Allonger la chaîne n'aide pas ici, puisque Optuna a choisi 145 itérations alors qu'il pouvait aller jusqu'à 512 : les tirages successifs d'une même chaîne se ressemblent. **Lancer plusieurs chaînes indépendantes** et moyenner leurs prédictions explore mieux la loi a posteriori. C'est la même méthode, mieux calculée.

### Le réglage

On garde le rang et le nombre d'itérations de la méthode 6. **Le seul hyperparamètre est le nombre de chaînes $C\in\{1,2,4,8\}$.** Comme il n'a que 4 valeurs, on les essaie toutes (grille complète, `GridSampler` d'Optuna) plutôt que de tirer 30 essais. Plus de chaînes coûte plus cher : $C$ fois le temps de la méthode 6.

In [ ]:
def bayes_chains(R, R_targets, n_chains):
    hp = RESULTS["Bayesian SVD++ flipped"]["best"]                 # rang et itérations de la méthode 6
    return np.mean([bayes_flipped(R, R_targets, **hp, seed=SEED + c) for c in range(n_chains)], 0)

run("Flipped, plusieurs chaînes", bayes_chains,
    lambda t: dict(n_chains=t.suggest_categorical("n_chains", [1, 2, 4, 8])),
    sampler=optuna.samplers.GridSampler({"n_chains": [1, 2, 4, 8]}), n_trials=4)

Flipped, plusieurs chaînes
  meilleurs hyperparamètres : {'n_chains': 8}
  RMSE validation (5 plis) : 0.8484 ± 0.0110
  RMSE test : 0.8501
  temps : recherche 11.0 min, apprentissage final 85 s


**Lecture :** Optuna retient **8 chaînes**, la plus grande valeur testée : 0,848 en validation, **0,850 en test**. C'est 0,0055 de mieux que la méthode 6 en test.
- Pli par pli (calcul fait juste en dessous), **le gain sur la méthode 6 est positif sur les 5 plis** : 0,0043 en moyenne, avec un écart-type de 0,0006. Le gain est petit mais certain.
- **C'est le même modèle, mieux calculé** : la méthode 6 gardait une part de hasard liée à sa chaîne MCMC.
- **Coût :** 8 fois celui de la méthode 6, soit 85 s pour l'apprentissage final sur le train. Sur train+test, matrice complète comprise, il faut 135 s (mesuré hors du notebook, avec le même RMSE de test).
- 8 étant la plus grande valeur de la grille, d'autres chaînes pourraient encore aider un peu, avec des gains de plus en plus faibles.

In [ ]:
# Méthode 7 contre méthode 6, pli par pli (mêmes plis : les difficultés propres à chaque pli s'annulent)
s6 = np.array([rmse(bayes_flipped(fit, val, **RESULTS["Bayesian SVD++ flipped"]["best"]), val) for fit, val in FOLDS])
s7 = np.array([rmse(bayes_chains(fit, val, **RESULTS["Flipped, plusieurs chaînes"]["best"]), val) for fit, val in FOLDS])
g = s6 - s7
print("gain 7 sur 6, pli par pli : " + " ".join(f"{x:+.4f}" for x in g) + f" | moyen {g.mean():.4f}, écart-type {g.std():.4f}")

gain 7 sur 6, pli par pli : +0.0033 +0.0041 +0.0050 +0.0047 +0.0043 | moyen 0.0043, écart-type 0.0006


## 11. Résultats

In [ ]:
print(f"{'Méthode':<28} {'RMSE val (5 plis)':>18} {'RMSE test':>10}   meilleurs hyperparamètres")
for name, r in RESULTS.items():
    print(f"{name:<28} {r['val']:>11.4f} ± {r['std']:.4f} {r['test']:>10.4f}   {r['best']}")

Méthode                       RMSE val (5 plis)  RMSE test   meilleurs hyperparamètres
Voisinage (cours)                 0.9944 ± 0.0090     1.0060   {'k': 144, 'sim': 'jaccard'}
Factorisation (cours)             0.9049 ± 0.0086     0.9123   {'k': 1, 'lam': 0.014684803764875042, 'mu': 0.9492861281053446, 'eta': 0.0002730449359463328, 'n_iter': 694}
SVD++ (Koren 2008)                0.8712 ± 0.0107     0.8767   {'k': 5, 'lr': 0.026036270993428392, 'reg_b': 0.10947980755691644, 'reg': 0.19484181194855843, 'n_epochs': 49}
Factorisation bayésienne          0.8636 ± 0.0110     0.8667   {'rank': 16, 'n_iter': 128}
Bayesian SVD++ (article)          0.8635 ± 0.0098     0.8667   {'rank': 32, 'n_iter': 246}
Bayesian SVD++ flipped            0.8527 ± 0.0106     0.8556   {'rank': 16, 'n_iter': 145}
Flipped, plusieurs chaînes        0.8484 ± 0.0110     0.8501   {'n_chains': 8}


**Le gain de la méthode 6 est-il réel ?** Il vaut 0,011, à peu près l'écart-type entre plis (0,011). Mais cet écart-type mélange deux choses : les plis n'ont pas tous la même difficulté, et cette difficulté touche toutes les méthodes de la même façon. Pour comparer deux méthodes, on regarde donc leur différence **pli par pli**.

In [ ]:
# Méthodes 4 et 6 comparées pli par pli, avec leurs meilleurs réglages (mêmes plis : les difficultés propres à chaque pli s'annulent)
gains = []
for f, (fit, val) in enumerate(FOLDS):
    a = rmse(bayes_mf(fit, val, **RESULTS["Factorisation bayésienne"]["best"]), val)
    b = rmse(bayes_flipped(fit, val, **RESULTS["Bayesian SVD++ flipped"]["best"]), val)
    gains.append(a - b)
    print(f"pli {f} : méthode 4 {a:.4f} | méthode 6 {b:.4f} | gain {a - b:+.4f}")
print(f"gain moyen {np.mean(gains):.4f}, écart-type du gain {np.std(gains):.4f}, gain positif sur {sum(g > 0 for g in gains)} plis sur 5")

pli 0 : méthode 4 0.8784 | méthode 6 0.8630 | gain +0.0153


pli 1 : méthode 4 0.8615 | méthode 6 0.8614 | gain +0.0001


pli 2 : méthode 4 0.8701 | méthode 6 0.8542 | gain +0.0159


pli 3 : méthode 4 0.8452 | méthode 6 0.8333 | gain +0.0120


pli 4 : méthode 4 0.8630 | méthode 6 0.8518 | gain +0.0112
gain moyen 0.0109, écart-type du gain 0.0057, gain positif sur 5 plis sur 5


Pli par pli, **la méthode 6 bat la méthode 4 sur les 5 plis** (gain moyen 0,011, écart-type du gain 0,006). La comparaison de la méthode 7 contre la méthode 6 est dans la section 10.

**Bilan**

| Méthode | Ce qu'elle ajoute | RMSE validation (5 plis) | RMSE test |
|---|---|---|---|
| 1. Voisinage (cours) | — | 0,994 ± 0,009 | 1,006 |
| 2. Factorisation (cours) | des facteurs latents | 0,905 ± 0,009 | 0,912 |
| 3. SVD++ (Koren 2008) | biais + implicite côté utilisateur | 0,871 ± 0,011 | 0,877 |
| 4. Factorisation bayésienne | biais + apprentissage bayésien | 0,864 ± 0,011 | 0,867 |
| 5. Bayesian SVD++ (article) | 3 et 4 réunis | 0,864 ± 0,010 | 0,867 |
| 6. Bayesian SVD++ flipped | 5 + implicite côté film | 0,853 ± 0,011 | 0,856 |
| **7. Flipped, 8 chaînes** | 6, calculée avec 8 chaînes MCMC | **0,848 ± 0,011** | **0,850** |

1. **Le classement est le même en validation et en test.** Les écarts entre méthodes voisines sont confirmés pli par pli :
   - 6 sur 4 : 5 plis sur 5 ;
   - 7 sur 6 : 5 plis sur 5.
2. **Ce que chaque ingrédient apporte :**
   - les **biais** : le grand saut pour la factorisation (0,912 → 0,877) ;
   - l'**apprentissage bayésien** : 0,010 ;
   - l'**implicite côté utilisateur** : rien ;
   - l'**implicite côté film** : 0,011 ;
   - **plusieurs chaînes MCMC** : 0,004 à 0,005.
3. **Le choix recommandé est la méthode 7.** C'est un seul modèle, avec un gain sur la méthode 6 certain sur les 5 plis.
4. **Une seule idée explique les résultats : l'information et la régularisation doivent aller là où les données manquent.** Ici, ce sont les films (3 notes en médiane), pas les utilisateurs (24). C'est ce que fait l'implicite côté film.
5. **Le RMSE de validation est un peu plus bas que celui du test**, pour toutes les méthodes. La note réservée de chaque film reste toujours en apprentissage : les films très peu notés, les plus difficiles, apparaissent donc rarement en validation, alors qu'ils sont dans le test. Comme toutes les méthodes sont mesurées sur les mêmes cases, la comparaison reste juste.

*Les méthodes ont été exécutées en plusieurs fois (1, 2 et 5 ; 3 et 4 ; 6 et la comparaison 4/6 ; 7), avec le même code de préparation et les mêmes graines. Deux fonctions ont reçu un argument optionnel après coup : `run` (échantillonneur et nombre d'essais) et `bayes_flipped` (graine). Leur comportement par défaut est inchangé. Tout est déterministe : le notebook relancé d'un bout à l'autre donne les mêmes chiffres, en environ 5 h de calcul.*